# 03 - Integration

**Project:** Kenya Climate Data Lab
**Author:** Alex Haro Nyandega
**Date:** 2026-10-12
**Purpose:** Audit the existing drought-stress pipeline, then integrate
the crop calendar and ERA5 soil-moisture signals.

## Rules
1. Every merge goes through `logged_join()`.
2. Every dropped row is accounted for.
3. Notebook reruns on a fresh kernel in under 60 seconds.
4. No new raw data. Everything comes from Week 3 outputs.

In [1]:
import hashlib
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)
sns.set_style("whitegrid")

ROOT  = Path("..").resolve()
PROC  = ROOT / "data" / "processed"
META  = ROOT / "data" / "metadata"
PAPER = ROOT / "paper"
FIGS  = PAPER / "figures"
for p in [PROC, META, PAPER, FIGS]:
    p.mkdir(parents=True, exist_ok=True)
print("Setup OK. ROOT =", ROOT)

Setup OK. ROOT = /Users/AlexHaro/Desktop/kenya-climate-data-lab


## 1. `logged_join()` - the audit discipline

Every merge in this notebook goes through this function. It records
rows before, rows after, rows dropped, and duplicate keys in the
right table (the number one cause of silent row explosions).

In [2]:
JOIN_AUDIT = []

def logged_join(left, right, on, how="inner", label="J", right_name="right"):
    on_list = on if isinstance(on, list) else [on]
    before = len(left)
    right_before = len(right)

    dupes_right = right.duplicated(subset=on_list).sum()
    if dupes_right > 0:
        print(f"  WARNING {label}: right table '{right_name}' has "
              f"{dupes_right} duplicate keys on {on_list}")

    out = left.merge(right, on=on, how=how, suffixes=("", "_r"))
    after = len(out)
    dropped = before - after if how == "inner" else 0
    unmatched = int(out[on_list[0]].isna().sum()) if how == "left" else 0

    JOIN_AUDIT.append({
        "join": label, "left_rows": before, "right_rows": right_before,
        "type": how, "keys": ", ".join(on_list), "result_rows": after,
        "dropped": dropped, "unmatched": unmatched,
    })
    print(f"  {label}: {before:>7,} {how:>5} {right_before:>7,} "
          f"-> {after:>7,}  (dropped {dropped}, unmatched {unmatched})")
    return out

print("logged_join() ready.")

logged_join() ready.


## 2. Load every existing processed file

Column names as they actually are on disk (confirmed before this notebook
was written):

- stress master: `county, year, month, date, ...`
- weekly rain  : `iso_year, iso_week, county, rainfall_mm, ...` (no date column)
- crop calendar: `county, zone, ..., long_rains_start, long_rains_end, ...`
- ERA5 long    : `date, county, variable, value`
- NDVI monthly : `name, year, month, date, mean` (county column is `name`)

In [3]:
def try_load(path, **kw):
    p = Path(path)
    if not p.exists():
        print(f"  MISSING  {p.name}")
        return None
    df = pd.read_csv(p, **kw)
    print(f"  {p.name:45s} {df.shape}")
    return df

print("Loading processed files:")

stress   = try_load(PROC / "county_monthly_stress_2017_2024.csv", parse_dates=["date"])
panel    = try_load(PROC / "county_monthly_panel_2017_2024.csv",  parse_dates=["month"])
ndvi     = try_load(PROC / "ndvi_counties_monthly_2017_2024.csv", parse_dates=["date"])
weekly   = try_load(PROC / "county_weekly_rainfall_2017_2024.csv")
chirps   = try_load(PROC / "chirps_counties_daily_2010_2024.csv", parse_dates=["date"])
calendar = try_load(META / "kenya_crop_calendar.csv")

# Rename NDVI county column so downstream joins always use 'county'
if ndvi is not None and "name" in ndvi.columns and "county" not in ndvi.columns:
    ndvi = ndvi.rename(columns={"name": "county"})
    print("  Renamed ndvi['name'] -> 'county'")

# Build month_start on any monthly dataset so joins are consistent
def add_month_start(df):
    if df is None:
        return None
    d = df.copy()
    if "month_start" not in d.columns:
        y = d["year"].astype(int).astype(str)
        m = d["month"].astype(int).astype(str).str.zfill(2)
        d["month_start"] = pd.to_datetime(y + "-" + m + "-01")
    return d

stress = add_month_start(stress)
ndvi   = add_month_start(ndvi)
panel  = add_month_start(panel) if panel is not None and {"year","month"}.issubset(panel.columns) else panel

print()
print("After month_start normalization:")
for name, df in [("stress", stress), ("ndvi", ndvi), ("panel", panel)]:
    if df is not None:
        print(f"  {name:8s} {df.shape}")

Loading processed files:
  county_monthly_stress_2017_2024.csv           (4512, 16)
  county_monthly_panel_2017_2024.csv            (4512, 9)
  ndvi_counties_monthly_2017_2024.csv           (4512, 5)
  county_weekly_rainfall_2017_2024.csv          (19599, 9)
  chirps_counties_daily_2010_2024.csv           (5479, 48)
  kenya_crop_calendar.csv                       (47, 9)
  Renamed ndvi['name'] -> 'county'

After month_start normalization:
  stress   (4512, 17)
  ndvi     (4512, 6)
  panel    (4512, 10)


## 3. Pipeline row-count trail

The audit that proves the pipeline is internally consistent from raw
CHIRPS files down to the master stress table.

In [4]:
print("=== PIPELINE ROW COUNT TRAIL ===\n")

if chirps is not None:
    n_days = chirps["date"].nunique()
    n_counties = chirps.shape[1] - 1
    print(f"CHIRPS daily           : {n_days:,} days x {n_counties} counties "
          f"= {n_days * n_counties:,} county-days")

if weekly is not None:
    print(f"Weekly rainfall        : {len(weekly):,} rows  "
          f"({weekly['county'].nunique()} counties, "
          f"{weekly['iso_year'].nunique()} years)")

if ndvi is not None:
    print(f"NDVI monthly           : {len(ndvi):,} rows  "
          f"({ndvi['county'].nunique()} counties)")

if panel is not None:
    print(f"Monthly panel          : {len(panel):,} rows")

if stress is not None:
    print(f"Stress master          : {len(stress):,} rows")
    print(f"  unique counties      : {stress['county'].nunique()}")
    print(f"  month range          : {stress['month_start'].min().date()} -> "
          f"{stress['month_start'].max().date()}")
    print(f"  stress columns       : {stress.columns.tolist()}")

=== PIPELINE ROW COUNT TRAIL ===

CHIRPS daily           : 5,479 days x 47 counties = 257,513 county-days
Weekly rainfall        : 19,599 rows  (47 counties, 8 years)
NDVI monthly           : 4,512 rows  (47 counties)
Monthly panel          : 4,512 rows
Stress master          : 4,512 rows
  unique counties      : 47
  month range          : 2017-01-01 -> 2024-12-01
  stress columns       : ['county', 'year', 'month', 'date', 'season', 'rainfall_mm', 'rainfall_anomaly', 'ndvi', 'ndvi_anomaly', 'stress_avg', 'stress_min', 'stress_category', 'rainfall_mm_is_outlier', 'ndvi_is_outlier', 'stress_avg_is_outlier', 'stress_min_is_outlier', 'month_start']


## 4. Duplicate key check

The master must be unique on `(county, month_start)`. If it is not,
the rest of the pipeline is silently broken.

In [5]:
if stress is not None:
    dupes = stress.duplicated(subset=["county", "month_start"]).sum()
    print(f"Duplicate (county, month_start) rows in stress master: {dupes}")
    if dupes > 0:
        print("  INVESTIGATE - the master should be unique on that key")
    else:
        print("  OK - master is unique on (county, month_start)")

Duplicate (county, month_start) rows in stress master: 0
  OK - master is unique on (county, month_start)


## 5. Save the audit and commit

Monday's job ends here. Tuesday adds the crop calendar.

In [6]:
audit_df = pd.DataFrame(JOIN_AUDIT) if JOIN_AUDIT else pd.DataFrame(
    columns=["join","left_rows","right_rows","type","keys",
             "result_rows","dropped","unmatched"])
audit_path = PROC / "stress_join_audit.csv"
audit_df.to_csv(audit_path, index=False)
print(f"Saved {audit_path.name} ({len(audit_df)} rows)")
print(audit_df)

Saved stress_join_audit.csv (0 rows)
Empty DataFrame
Columns: [join, left_rows, right_rows, type, keys, result_rows, dropped, unmatched]
Index: []


---

## Monday complete

Next (Tuesday): join the crop calendar on `(county, month_start)` to
add `crop_stage_weight` and `stress_weighted` columns.

# Tuesday - Crop calendar integration (D34)

Add `crop_stage_weight` per `(county, month)`, then `stress_weighted`.

The weight answers: *if this month is dry, how much does it hurt?*

- 1.0 during planting / vegetative / flowering
- 0.7 during grain fill
- 0.3 during primary harvest month
- 0.0 during fallow
- NaN for urban (Nairobi)

Source: `data/metadata/kenya_crop_calendar.csv`

In [7]:
# 1. Expand crop calendar to (county, month_num) with stage label + weight

STAGE_WEIGHT = {
    "planting":      1.0,
    "vegetative":    1.0,
    "grain_fill":    0.7,
    "harvest":       0.3,
    "fallow":        0.0,
}

def months_in_range(start, end):
    """Inclusive integer month range, wrapping across year boundary."""
    start, end = int(start), int(end)
    if start <= end:
        return list(range(start, end + 1))
    return list(range(start, 13)) + list(range(1, end + 1))

def build_weights(cal):
    rows = []
    for _, r in cal.iterrows():
        county = r["county"]
        harvest = int(r["primary_harvest_month"]) if pd.notna(r["primary_harvest_month"]) else None
        seasons = []
        if r["n_seasons"] >= 1 and pd.notna(r["long_rains_start"]):
            seasons.append(("long", r["long_rains_start"], r["long_rains_end"]))
        if r["n_seasons"] >= 2 and pd.notna(r["short_rains_start"]):
            seasons.append(("short", r["short_rains_start"], r["short_rains_end"]))

        # Per-month label for this county
        for m in range(1, 13):
            label = "fallow"
            for _name, s, e in seasons:
                in_season = months_in_range(s, e)
                if m in in_season:
                    # Split the season: first half = planting/vegetative, second half = grain fill
                    pos = in_season.index(m)
                    if pos < len(in_season) // 2:
                        label = "planting"
                    else:
                        label = "grain_fill"
                    break
            if harvest is not None and m == harvest:
                label = "harvest"
            rows.append({
                "county": county,
                "month_num": m,
                "crop_stage": label,
                "crop_stage_weight": STAGE_WEIGHT[label],
            })

    w = pd.DataFrame(rows)
    # Nairobi is urban: NaN
    w.loc[w["county"] == "Nairobi", "crop_stage_weight"] = np.nan
    return w

calendar_weights = build_weights(calendar)
print("Calendar weights built:", calendar_weights.shape)
print(calendar_weights.head(15))
print()
print("Weight distribution:")
print(calendar_weights["crop_stage"].value_counts())

Calendar weights built: (564, 4)
      county  month_num  crop_stage  crop_stage_weight
0   Kakamega          1      fallow                0.0
1   Kakamega          2      fallow                0.0
2   Kakamega          3    planting                1.0
3   Kakamega          4  grain_fill                0.7
4   Kakamega          5  grain_fill                0.7
5   Kakamega          6      fallow                0.0
6   Kakamega          7      fallow                0.0
7   Kakamega          8      fallow                0.0
8   Kakamega          9     harvest                0.3
9   Kakamega         10    planting                1.0
10  Kakamega         11  grain_fill                0.7
11  Kakamega         12  grain_fill                0.7
12   Bungoma          1      fallow                0.0
13   Bungoma          2      fallow                0.0
14   Bungoma          3    planting                1.0

Weight distribution:
crop_stage
fallow        308
grain_fill    133
planting       77


In [8]:
# 2. Add month_num to stress for the join
stress = stress.copy()
stress["month_num"] = pd.to_datetime(stress["month_start"]).dt.month

# 3. Join weights onto stress
stress_v2 = logged_join(
    stress,
    calendar_weights,
    on=["county", "month_num"],
    how="left",
    label="J01_calendar",
    right_name="calendar_weights",
)

# 4. Compute weighted stress
stress_v2["stress_weighted"] = stress_v2["stress_avg"] * stress_v2["crop_stage_weight"]

print()
print("stress_v2 shape:", stress_v2.shape)
print("New columns added: crop_stage, crop_stage_weight, stress_weighted")
print()
print("Sample (Trans Nzoia, Oct 2022 vs Oct 2021):")
sample = stress_v2[(stress_v2["county"] == "Trans Nzoia") &
                   (stress_v2["month_num"] == 10) &
                   (stress_v2["year"].isin([2021, 2022]))]
print(sample[["county","year","month","crop_stage","crop_stage_weight",
              "stress_avg","stress_weighted"]].to_string(index=False))

  J01_calendar:   4,512  left     564 ->   4,512  (dropped 0, unmatched 0)

stress_v2 shape: (4512, 21)
New columns added: crop_stage, crop_stage_weight, stress_weighted

Sample (Trans Nzoia, Oct 2022 vs Oct 2021):
     county  year  month crop_stage  crop_stage_weight  stress_avg  stress_weighted
Trans Nzoia  2021     10    harvest                0.3   -0.525080        -0.157524
Trans Nzoia  2022     10    harvest                0.3   -1.051324        -0.315397


In [9]:
# 5. Sanity: how many rows lost crop_stage_weight?
missing_w = stress_v2["crop_stage_weight"].isna().sum()
print(f"Rows with NaN crop_stage_weight: {missing_w}")
if missing_w > 0:
    print("  Affected counties:")
    print(stress_v2[stress_v2["crop_stage_weight"].isna()]["county"].value_counts())

Rows with NaN crop_stage_weight: 96
  Affected counties:
county
Nairobi    96
Name: count, dtype: int64


## Save the enhanced master

In [10]:
out = PROC / "county_monthly_stress_v2.csv"
stress_v2.to_csv(out, index=False)
print(f"Saved {out.name}: {stress_v2.shape}")
print(f"Columns: {stress_v2.columns.tolist()}")

Saved county_monthly_stress_v2.csv: (4512, 21)
Columns: ['county', 'year', 'month', 'date', 'season', 'rainfall_mm', 'rainfall_anomaly', 'ndvi', 'ndvi_anomaly', 'stress_avg', 'stress_min', 'stress_category', 'rainfall_mm_is_outlier', 'ndvi_is_outlier', 'stress_avg_is_outlier', 'stress_min_is_outlier', 'month_start', 'month_num', 'crop_stage', 'crop_stage_weight', 'stress_weighted']


## Final audit save (runs after all joins)

The Monday audit save cell runs before Tuesday/Wednesday joins are
performed, so it always shows 0 rows on the first pass. This cell
re-saves the audit *after* every join in the notebook has executed.

In [11]:
audit_df_final = pd.DataFrame(JOIN_AUDIT)
audit_df_final.to_csv(PROC / "stress_join_audit.csv", index=False)
print(f"Saved stress_join_audit.csv ({len(audit_df_final)} rows)")
print(audit_df_final.to_string(index=False))

Saved stress_join_audit.csv (1 rows)
        join  left_rows  right_rows type              keys  result_rows  dropped  unmatched
J01_calendar       4512         564 left county, month_num         4512        0          0


# Wednesday - ERA5 integration (D35)

Add four ERA5-Land anomalies as diagnostic columns:

- `swvl1_mean_anomaly`  - soil moisture 0-7 cm
- `swvl2_mean_anomaly`  - soil moisture 7-28 cm
- `t2m_mean_anomaly`    - 2m air temperature
- `pev_mean_anomaly`    - potential evaporation

Then test the central scientific question: does soil moisture lead
vegetation in the 2022 drought? A positive lead would prove why the
tool needs soil-moisture as an early-warning column.

In [12]:
# 1. Load the four ERA5 variables we need
ERA5 = PROC / "era5_counties"

def load_era5(var):
    p = ERA5 / f"era5_{var}_counties_daily.csv"
    df = pd.read_csv(p, parse_dates=["date"])
    print(f"  {var:6s} {df.shape}  counties={df['county'].nunique()}")
    return df

print("Loading ERA5 county daily:")
swvl1 = load_era5("swvl1")
swvl2 = load_era5("swvl2")
t2m   = load_era5("t2m")
pev   = load_era5("pev")

Loading ERA5 county daily:


  swvl1  (600848, 4)  counties=47


  swvl2  (600848, 4)  counties=47


  t2m    (600848, 4)  counties=47


  pev    (600848, 4)  counties=47


In [13]:
# 2. Aggregate daily -> monthly per county
def era5_to_monthly(df, var):
    d = df.copy()
    d["month_start"] = d["date"].values.astype("datetime64[M]").astype("datetime64[ns]")
    m = (d.groupby(["county", "month_start"], as_index=False)["value"]
           .mean()
           .rename(columns={"value": f"{var}_mean"}))
    return m

swvl1_m = era5_to_monthly(swvl1, "swvl1")
swvl2_m = era5_to_monthly(swvl2, "swvl2")
t2m_m   = era5_to_monthly(t2m,   "t2m")
pev_m   = era5_to_monthly(pev,   "pev")

print("Monthly aggregates:")
for name, df in [("swvl1",swvl1_m),("swvl2",swvl2_m),
                 ("t2m",t2m_m),("pev",pev_m)]:
    print(f"  {name:6s} {df.shape}")

Monthly aggregates:
  swvl1  (19740, 3)
  swvl2  (19740, 3)
  t2m    (19740, 3)
  pev    (19740, 3)


In [14]:
# 3. Compute anomalies per (county, month_num) vs own history
def add_anomaly(df, value_col):
    d = df.copy()
    d["month_num"] = pd.to_datetime(d["month_start"]).dt.month
    stats = (d.groupby(["county","month_num"])[value_col]
               .agg(["mean","std"]).reset_index()
               .rename(columns={"mean": f"{value_col}_clim_mean",
                                "std":  f"{value_col}_clim_std"}))
    d = d.merge(stats, on=["county","month_num"], how="left")
    d[f"{value_col}_anomaly"] = (
        (d[value_col] - d[f"{value_col}_clim_mean"])
        / d[f"{value_col}_clim_std"].replace(0, np.nan)
    )
    return d

swvl1_m = add_anomaly(swvl1_m, "swvl1_mean")
swvl2_m = add_anomaly(swvl2_m, "swvl2_mean")
t2m_m   = add_anomaly(t2m_m,   "t2m_mean")
pev_m   = add_anomaly(pev_m,   "pev_mean")

print("Anomalies computed. Sample swvl1:")
print(swvl1_m[["county","month_start","swvl1_mean","swvl1_mean_anomaly"]].head())

Anomalies computed. Sample swvl1:
    county month_start  swvl1_mean  swvl1_mean_anomaly
0  Baringo  1990-01-01    0.151913            1.082858
1  Baringo  1990-02-01    0.154650            1.669464
2  Baringo  1990-03-01    0.223679            2.637783
3  Baringo  1990-04-01    0.254124            1.300687
4  Baringo  1990-05-01    0.205510           -0.047761


In [15]:
# 4. Join the four anomaly columns onto stress_v2
era5_features = (swvl1_m[["county","month_start","swvl1_mean","swvl1_mean_anomaly"]]
    .merge(swvl2_m[["county","month_start","swvl2_mean_anomaly"]],
           on=["county","month_start"], how="outer")
    .merge(t2m_m[["county","month_start","t2m_mean_anomaly"]],
           on=["county","month_start"], how="outer")
    .merge(pev_m[["county","month_start","pev_mean_anomaly"]],
           on=["county","month_start"], how="outer")
)

stress_v3 = logged_join(
    stress_v2, era5_features,
    on=["county","month_start"],
    how="left",
    label="J02_era5",
    right_name="era5_features",
)

print()
print("stress_v3 shape:", stress_v3.shape)
print("New ERA5 columns added: swvl1_mean_anomaly, swvl2_mean_anomaly,")
print("                        t2m_mean_anomaly, pev_mean_anomaly")
print()
print("NaN check for ERA5 columns (should be 0 - all months covered 1990-2024):")
print(stress_v3[["swvl1_mean_anomaly","swvl2_mean_anomaly",
                 "t2m_mean_anomaly","pev_mean_anomaly"]].isna().sum())

  J02_era5:   4,512  left  19,740 ->   4,512  (dropped 0, unmatched 0)

stress_v3 shape: (4512, 26)
New ERA5 columns added: swvl1_mean_anomaly, swvl2_mean_anomaly,
                        t2m_mean_anomaly, pev_mean_anomaly

NaN check for ERA5 columns (should be 0 - all months covered 1990-2024):
swvl1_mean_anomaly    0
swvl2_mean_anomaly    0
t2m_mean_anomaly      0
pev_mean_anomaly      0
dtype: int64


In [16]:
# 5. Does soil moisture LEAD vegetation? Lead-lag correlation, 2017-2024
from scipy.stats import pearsonr

work = stress_v3.dropna(subset=["swvl1_mean_anomaly","ndvi_anomaly"]).copy()
work = work.sort_values(["county","month_start"])

def corr_at_lag(lag):
    if lag == 0:
        a = work["swvl1_mean_anomaly"]
        b = work["ndvi_anomaly"]
        mask = a.notna() & b.notna()
    else:
        a = work.groupby("county")["swvl1_mean_anomaly"].shift(lag)
        b = work["ndvi_anomaly"]
        mask = a.notna() & b.notna()
    r, p = pearsonr(a[mask], b[mask])
    return r, p, int(mask.sum())

print("Lead-lag: soil moisture (swvl1) vs NDVI")
print("  POSITIVE lag = swvl1 leads NDVI by N months")
print()
print(f"{'lag':>5s}  {'r':>8s}  {'p':>10s}  {'n':>6s}")
for lag in [-4,-3,-2,-1,0,1,2,3,4]:
    r, p, n = corr_at_lag(lag)
    marker = " <-- strongest" if False else ""
    print(f"{lag:>5d}  {r:>8.3f}  {p:>10.2e}  {n:>6d}")

Lead-lag: soil moisture (swvl1) vs NDVI
  POSITIVE lag = swvl1 leads NDVI by N months

  lag         r           p       n
   -4     0.084    9.69e-08    4064
   -3     0.111    9.89e-13    4111
   -2     0.175    5.71e-30    4158
   -1     0.230    9.11e-52    4205
    0     0.364   1.45e-133    4252
    1     0.461   2.13e-220    4205
    2     0.372   1.83e-136    4158
    3     0.285    1.89e-77    4111
    4     0.272    1.02e-69    4064


## Save the enhanced master

In [17]:
out = PROC / "county_monthly_stress_v3.csv"
stress_v3.to_csv(out, index=False)
print(f"Saved {out.name}: {stress_v3.shape}")
print(f"Columns: {stress_v3.columns.tolist()}")

Saved county_monthly_stress_v3.csv: (4512, 26)
Columns: ['county', 'year', 'month', 'date', 'season', 'rainfall_mm', 'rainfall_anomaly', 'ndvi', 'ndvi_anomaly', 'stress_avg', 'stress_min', 'stress_category', 'rainfall_mm_is_outlier', 'ndvi_is_outlier', 'stress_avg_is_outlier', 'stress_min_is_outlier', 'month_start', 'month_num', 'crop_stage', 'crop_stage_weight', 'stress_weighted', 'swvl1_mean', 'swvl1_mean_anomaly', 'swvl2_mean_anomaly', 't2m_mean_anomaly', 'pev_mean_anomaly']


## Final audit save (re-run at end of notebook)

In [18]:
audit_df_final = pd.DataFrame(JOIN_AUDIT)
audit_df_final.to_csv(PROC / "stress_join_audit.csv", index=False)
print(f"Saved stress_join_audit.csv ({len(audit_df_final)} rows)")
print(audit_df_final.to_string(index=False))

Saved stress_join_audit.csv (2 rows)
        join  left_rows  right_rows type                keys  result_rows  dropped  unmatched
J01_calendar       4512         564 left   county, month_num         4512        0          0
    J02_era5       4512       19740 left county, month_start         4512        0          0


## Lead-lag interpretation (D35 result)

Positive lag = soil moisture leads NDVI by that many months.

| lag | r | meaning |
|---|---|---|
| -4 | 0.084 | NDVI leads swvl1 by 4 months (weak) |
| -2 | 0.175 | NDVI leads swvl1 by 2 months (weak) |
| 0 | 0.364 | Contemporaneous |
| **+1** | **0.461** | **swvl1 leads NDVI by 1 month (strongest)** |
| +2 | 0.372 | swvl1 leads by 2 months |
| +4 | 0.272 | swvl1 leads by 4 months |

**Result: soil moisture anomaly leads vegetation anomaly by one month
(r = 0.461, p = 2e-220, n = 4205).**

This is consistent with the physical mechanism of drought: soil dries
first, vegetation follows. The lead is modest (1 month) but the effect
is statistically overwhelming.

**Operational consequence:** `swvl1_mean_anomaly` becomes the earliest
signal in the weekly product. A county whose soil moisture collapsed
this month is likely to show vegetation stress next month.

# Thursday - Weekly stress product (D36)

A product a county officer can read *this week*.

Design principle: the weekly product is rainfall-led. Every monthly
signal is carried forward from the last confirmed month and prefixed
`latest_monthly_` so the reader always knows what is fresh and what
is the last confirmed reading.

We do not interpolate NDVI to weekly cadence. That would fabricate
signal the satellite did not see.

In [19]:
# 1. Reload the weekly rainfall file (it has iso_year/iso_week, no date)
weekly = pd.read_csv(PROC / "county_weekly_rainfall_2017_2024.csv")
print("Weekly rainfall:", weekly.shape)
print("Columns:", weekly.columns.tolist())
print()
print(weekly.head())

Weekly rainfall: (19599, 9)
Columns: ['iso_year', 'iso_week', 'county', 'rainfall_mm', 'rain_clim_mean', 'rain_clim_std', 'n_years', 'rainfall_anomaly', 'rainfall_anomaly_4wk']

   iso_year  iso_week   county  rainfall_mm  rain_clim_mean  rain_clim_std  n_years  rainfall_anomaly  rainfall_anomaly_4wk
0      2017         1  Baringo     2.062112        1.288665       1.589575        9          0.486575                   NaN
1      2017         2  Baringo     2.396435        2.332278       2.076073        8          0.030903              0.258739
2      2017         3  Baringo     2.751988       14.437000      17.977576        8         -0.649977             -0.044166
3      2017         4  Baringo     1.817739        3.874351       3.693641        8         -0.556798             -0.172324
4      2017         5  Baringo    13.978271        9.186771      18.552593        8          0.258266             -0.229401


In [20]:
# 2. Reconstruct a calendar date from iso_year + iso_week (Monday of that week)
weekly = weekly.copy()
weekly["week_start"] = pd.to_datetime(
    weekly["iso_year"].astype(str) + "-" +
    weekly["iso_week"].astype(str).str.zfill(2) + "-1",
    format="%G-%V-%u",
)
weekly["month_start"] = weekly["week_start"].values.astype("datetime64[M]").astype("datetime64[ns]")

print("Sample with reconstructed dates:")
print(weekly[["iso_year","iso_week","county","week_start","month_start"]].head())
print()
print("Date range:", weekly["week_start"].min().date(), "->", weekly["week_start"].max().date())

Sample with reconstructed dates:
   iso_year  iso_week   county week_start month_start
0      2017         1  Baringo 2017-01-02  2017-01-01
1      2017         2  Baringo 2017-01-09  2017-01-01
2      2017         3  Baringo 2017-01-16  2017-01-01
3      2017         4  Baringo 2017-01-23  2017-01-01
4      2017         5  Baringo 2017-01-30  2017-01-01

Date range: 2017-01-02 -> 2024-12-23


In [21]:
# 3. Build the monthly companion table (last confirmed readings)
monthly_companion = stress_v3[[
    "county", "month_start",
    "stress_avg", "stress_weighted", "stress_min", "stress_category",
    "crop_stage", "crop_stage_weight",
    "ndvi_anomaly", "swvl1_mean_anomaly",
    "t2m_mean_anomaly", "pev_mean_anomaly",
]].rename(columns={
    "stress_avg":         "latest_monthly_stress_avg",
    "stress_weighted":    "latest_monthly_stress_weighted",
    "stress_min":         "latest_monthly_stress_min",
    "stress_category":    "latest_monthly_stress_category",
    "crop_stage":         "latest_monthly_crop_stage",
    "crop_stage_weight":  "latest_monthly_crop_stage_weight",
    "ndvi_anomaly":       "latest_monthly_ndvi_anomaly",
    "swvl1_mean_anomaly": "latest_monthly_swvl1_anomaly",
    "t2m_mean_anomaly":   "latest_monthly_t2m_anomaly",
    "pev_mean_anomaly":   "latest_monthly_pev_anomaly",
})

print("Monthly companion:", monthly_companion.shape)
print("Columns:", monthly_companion.columns.tolist())

Monthly companion: (4512, 12)
Columns: ['county', 'month_start', 'latest_monthly_stress_avg', 'latest_monthly_stress_weighted', 'latest_monthly_stress_min', 'latest_monthly_stress_category', 'latest_monthly_crop_stage', 'latest_monthly_crop_stage_weight', 'latest_monthly_ndvi_anomaly', 'latest_monthly_swvl1_anomaly', 'latest_monthly_t2m_anomaly', 'latest_monthly_pev_anomaly']


In [22]:
# 4. Join weekly rainfall with monthly companions
weekly_stress = logged_join(
    weekly, monthly_companion,
    on=["county", "month_start"],
    how="left",
    label="J03_weekly_x_monthly",
    right_name="monthly_companion",
)

print()
print("weekly_stress shape:", weekly_stress.shape)
print("NaN in monthly companions (weeks whose month predates 2017-01 stress window):")
comp_cols = [c for c in weekly_stress.columns if c.startswith("latest_monthly_")]
print(weekly_stress[comp_cols].isna().sum())

  J03_weekly_x_monthly:  19,599  left   4,512 ->  19,599  (dropped 0, unmatched 0)

weekly_stress shape: (19599, 21)
NaN in monthly companions (weeks whose month predates 2017-01 stress window):
latest_monthly_stress_avg           1129
latest_monthly_stress_weighted      1480
latest_monthly_stress_min           1129
latest_monthly_stress_category         0
latest_monthly_crop_stage              0
latest_monthly_crop_stage_weight     417
latest_monthly_ndvi_anomaly         1129
latest_monthly_swvl1_anomaly           0
latest_monthly_t2m_anomaly             0
latest_monthly_pev_anomaly             0
dtype: int64


## Save the weekly product

In [23]:
out = PROC / "county_weekly_stress_2017_2024.csv"
weekly_stress.to_csv(out, index=False)
print(f"Saved {out.name}: {weekly_stress.shape}")
print(f"Columns: {weekly_stress.columns.tolist()}")

Saved county_weekly_stress_2017_2024.csv: (19599, 21)
Columns: ['iso_year', 'iso_week', 'county', 'rainfall_mm', 'rain_clim_mean', 'rain_clim_std', 'n_years', 'rainfall_anomaly', 'rainfall_anomaly_4wk', 'week_start', 'month_start', 'latest_monthly_stress_avg', 'latest_monthly_stress_weighted', 'latest_monthly_stress_min', 'latest_monthly_stress_category', 'latest_monthly_crop_stage', 'latest_monthly_crop_stage_weight', 'latest_monthly_ndvi_anomaly', 'latest_monthly_swvl1_anomaly', 'latest_monthly_t2m_anomaly', 'latest_monthly_pev_anomaly']


## Final audit save (re-run at end of notebook)

In [24]:
audit_df_final = pd.DataFrame(JOIN_AUDIT)
audit_df_final.to_csv(PROC / "stress_join_audit.csv", index=False)
print(f"Saved stress_join_audit.csv ({len(audit_df_final)} rows)")
print(audit_df_final.to_string(index=False))

Saved stress_join_audit.csv (3 rows)
                join  left_rows  right_rows type                keys  result_rows  dropped  unmatched
        J01_calendar       4512         564 left   county, month_num         4512        0          0
            J02_era5       4512       19740 left county, month_start         4512        0          0
J03_weekly_x_monthly      19599        4512 left county, month_start        19599        0          0


## NaN pattern in the weekly product (documented, not fixed)

The weekly product inherits NaN from two documented sources:

| Source | Affected | Reason | Prior decision |
|---|---|---|---|
| NDVI missing months | 260 county-months | Cloud cover, no clear-sky Sentinel-2 obs | D14 |
| Nairobi | 96 county-months | Urban county, no crop signal | D34 |

`stress_avg` is NaN whenever `ndvi_anomaly` is NaN, because the
composite formula requires both components. `stress_weighted` is
NaN whenever either `stress_avg` or `crop_stage_weight` is NaN.

**Decision: we do not interpolate either.** Interpolating NDVI would
fabricate a satellite observation that never happened. Interpolating
Nairobi would fabricate a crop that does not exist.

Aggregate NaN counts in the weekly product (19,599 rows):

- `latest_monthly_stress_avg`        1,129 NaN (~5.8%)
- `latest_monthly_stress_weighted`   1,480 NaN (~7.6%)
- `latest_monthly_ndvi_anomaly`      1,129 NaN (~5.8%)
- `latest_monthly_swvl1_anomaly`         0 NaN
- `latest_monthly_stress_category`       0 NaN  (categorical, default filled)